# 04 — Retrieval recall (Week 2 dev slice)

Read-only on frozen assets and prior notebook/JSON outputs. Function-based cells (copy-pasteable to `src/` later).
- **Chunkers:** C2 (`fixed_char` 800/100), `section_tok512_ov10`, `table_tok512_fixed` (nb01 bug fixed in-notebook).
- **Indexes:** BM25 (`rank_bm25`) and dense (course model `bge-small-en-v1.5` via VM02, embeddings cached to disk), per chunker.
- **Metrics:** recall@1/3/5/10 and MRR for BM25, dense, hybrid RRF (k=60). Counts reported; no winner claims.
- **Eval:** larger dev slice (dev-answer gold evidence documents, not 16-doc sample) with gold evidence records; chunk-level doc/page matching (not full-excerpt containment).


In [2]:
# Parameters
FREEZE_PATH = "data/corpus/freeze_v1.json"
PAGE_DIR = "data/corpus/extract/page_extracts"
TABLE_DIR = "data/corpus/extract/table_records"
GOLD_PATH = "data/eval/datasets/gold_v2_eval_v8_public.jsonl"
CHUNKERS = ["C2", "section_tok512_ov10", "table_tok512_fixed"]
KS = (1, 3, 5, 10)
RRF_K = 60
EMBED_MODEL = "bge-small-en-v1.5"
EMBED_BATCH = 16
EMBED_MAX_RETRIES = 5
CACHE_ROOT = "results/week2_retrieval_cache"
RESULTS_JSON = "results/week2_retrieval.json"
ANALYSIS_MD = "docs/WEEK2_RETRIEVAL_ANALYSIS.md"
# Indexed dev slice: all document_ids cited by dev-answer evidence records (larger than 16-doc sample)


In [3]:
import os, sys, json, time, re, hashlib, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
from dotenv import load_dotenv

_root = Path.cwd().resolve()
if not (_root / "src").is_dir():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
load_dotenv(_root / ".env")

from rank_bm25 import BM25Okapi
from src.rag_ingestion.tokenizer import tokenize, TOKENIZER_VERSION

FREEZE = json.loads(Path(FREEZE_PATH).read_text(encoding="utf-8"))
FREEZE_SHA = FREEZE.get("manifest_sha256")
DOCS_BY_ID = {d["document_id"]: d for d in FREEZE.get("documents", [])}
print("freeze docs", len(DOCS_BY_ID), "sha", FREEZE_SHA, "tokenizer", TOKENIZER_VERSION)


freeze docs 1022 sha 899ac206a760af531e102e01daf99e4ff61d36cb6fc1e2e7a28cf01b094b5edd tokenizer tokenizer.v1


In [4]:
# --- IO and eval set helpers (function-based) ---

def read_jsonl(path):
    p = Path(path)
    if not p.is_file():
        return []
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]


def load_pages(doc_id):
    return read_jsonl(Path(PAGE_DIR) / f"{doc_id}.pages.jsonl")


def load_tables(doc_id):
    return read_jsonl(Path(TABLE_DIR) / f"{doc_id}.tables.jsonl")


def usable_pages(doc_id):
    return [r for r in load_pages(doc_id) if r.get("usable_as_evidence") and isinstance(r.get("pdf_page_index"), int)]


def load_gold(path=GOLD_PATH):
    return read_jsonl(path)


def dev_answer_rows_with_evidence(gold_rows):
    rows = []
    for r in gold_rows:
        if r.get("split") != "dev" or r.get("expected_behaviour") != "answer":
            continue
        ev = [e for e in (r.get("evidence") or []) if isinstance(e, dict) and e.get("document_id") and isinstance(e.get("pdf_page"), int)]
        if ev:
            rows.append(r)
    return rows


def gold_evidence_set(row):
    return {(e["document_id"], int(e["pdf_page"])) for e in (row.get("evidence") or []) if isinstance(e, dict) and e.get("document_id") and isinstance(e.get("pdf_page"), int)}


def dev_slice_doc_ids(eval_rows):
    ids = set()
    for r in eval_rows:
        for e in (r.get("evidence") or []):
            if isinstance(e, dict) and e.get("document_id"):
                ids.add(e["document_id"])
    return sorted(ids)


In [5]:
# --- chunking primitives (function-based) ---
_TOK_RE = re.compile(r"\(?-?\d[\d,]*(?:\.\d+)?\)?|[a-z0-9]+", re.IGNORECASE)
HEADING_MAXLEN = 80
STMT_KEYS = ("balance sheet", "profit and loss", "income statement", "cash flow",
             "statement of", "notes to", "segment", "equity", "financial statements")


def tok_spans(text):
    return [m.span() for m in _TOK_RE.finditer(text or "")]


def is_heading(line):
    t = (line or "").strip()
    if not t or len(t) > HEADING_MAXLEN:
        return False
    if t.isupper() and any(c.isalpha() for c in t):
        return True
    low = t.lower()
    if any(k in low for k in STMT_KEYS) and len(t.split()) <= 8:
        return True
    if t.endswith(":") and len(t.split()) <= 10:
        return True
    words = t.split()
    if 1 <= len(words) <= 8 and not any(ch.isdigit() for ch in t):
        caps = sum(1 for w in words if w[:1].isupper())
        if caps >= max(1, len(words) - 1):
            return True
    return False


def detect_sections(text):
    text = text or ""
    lines = []
    i = 0
    for part in text.split("\n"):
        s, e = i, i + len(part)
        lines.append((part, s, e))
        i = e + 1
    blocks = []
    cur_head = None
    cur_body_s = 0
    cur_body_e = 0
    have = False

    def flush(h, bs, be):
        if bs is None or be is None or be < bs:
            bs, be = 0, 0
        blocks.append((h, text[bs:be], bs, be))

    for ln, s, e in lines:
        if is_heading(ln):
            if have:
                flush(cur_head, cur_body_s, cur_body_e)
            cur_head = ln.strip()
            cur_body_s = min(e + 1, len(text))
            cur_body_e = cur_body_s
            have = True
        else:
            if not have:
                cur_head = None
                cur_body_s = s
                cur_body_e = e
                have = True
            else:
                cur_body_e = e
    if have:
        flush(cur_head, cur_body_s, cur_body_e)
    elif text:
        blocks.append((None, text, 0, len(text)))
    return blocks


def windows_by_chars(text, size=800, overlap=100):
    step = max(1, size - overlap)
    out, i, n = [], 0, len(text)
    while i < n:
        s, e = i, min(n, i + size)
        if (text[s:e] or "").strip():
            out.append((s, e))
        if e >= n:
            break
        i += step
    return out


def windows_by_spans(spans, size, overlap_frac):
    if not spans:
        return []
    step = max(1, size - int(round(size * overlap_frac)))
    out, i, n = [], 0, len(spans)
    while i < n:
        j = min(n, i + size)
        s, e = spans[i][0], spans[j - 1][1]
        if e > s:
            out.append((s, e))
        if j >= n:
            break
        i += step
    return out


def render_table(rec):
    parts = []
    cap = (rec.get("caption") or "").strip()
    if cap:
        parts.append("caption: " + cap)
    parts.append("statement_kind: " + str(rec.get("statement_kind") or ""))
    uh = (rec.get("unit_header") or "").strip()
    if uh:
        parts.append("unit_header: " + uh)
    parts.append("reporting_basis: " + str(rec.get("reporting_basis") or ""))
    cols = rec.get("columns") or []
    if cols:
        parts.append("columns: " + " | ".join(
            f"{c.get('column_id')}:{c.get('header_raw')}" + (f" <{c.get('header_path')}>" if c.get("header_path") else "")
            for c in cols))
    ordmap = {c.get("column_id"): (c.get("column_ordinal") if isinstance(c.get("column_ordinal"), int) else i)
              for i, c in enumerate(cols)}
    for row in rec.get("rows") or []:
        label = (row.get("row_label_raw") or "").strip()
        pairs = []
        for cell in sorted(row.get("cells") or [], key=lambda x: ordmap.get(x.get("column_id"), 0)):
            raw = str(cell.get("raw") or "").strip()
            if raw:
                pairs.append(f"{cell.get('column_id')}={raw}")
        if label and pairs:
            parts.append(label + ": " + "; ".join(pairs))
        elif pairs:
            parts.append("; ".join(pairs))
    return "\n".join(p for p in parts if p.strip())


def make_chunk(chunk_id, text, document_id, page_start, page_end, chunker, meta_extra=None):
    m = {"document_id": document_id, "page_start": page_start, "page_end": page_end, "chunker": chunker}
    if meta_extra:
        m.update(meta_extra)
    return {"chunk_id": chunk_id, "text": text, "meta": m}


In [6]:
# --- chunker implementations (function-based) ---

def chunk_c2(doc_id, pages):
    out = []
    for i, rec in enumerate(pages):
        text = rec.get("page_text") or ""
        page = rec.get("pdf_page_index")
        for j, (s, e) in enumerate(windows_by_chars(text, 800, 100)):
            out.append(make_chunk(f"{doc_id}::C2::p{page}::w{j}", text[s:e], doc_id, page, page, "C2", {"source": "page"}))
    return out


def chunk_section_tok512_ov10(doc_id, pages):
    out = []
    for rec in pages:
        text = rec.get("page_text") or ""
        page = rec.get("pdf_page_index")
        secs = detect_sections(text)
        k = 0
        for si, (head, body, bs, be) in enumerate(secs):
            if not (body or "").strip():
                continue
            b_spans = tok_spans(body)
            if len(b_spans) < 512:
                s, e = bs, be
                out.append(make_chunk(f"{doc_id}::section_tok512_ov10::p{page}::s{si}::w{k}", text[s:e], doc_id, page, page, "section_tok512_ov10", {"source": "section", "heading": head}))
                k += 1
            else:
                for ls, le in windows_by_spans(b_spans, 512, 0.10):
                    s, e = bs + ls, bs + le
                    out.append(make_chunk(f"{doc_id}::section_tok512_ov10::p{page}::s{si}::w{k}", text[s:e], doc_id, page, page, "section_tok512_ov10", {"source": "section", "heading": head}))
                    k += 1
    return out


def chunk_table_tok512_fixed(doc_id, pages, tables):
    """Fixed (in-notebook) table-aware chunker:
    - emits one rendered chunk per table record (page_start..page_end)
    - emits prose token windows (512/10%) only for pages NOT covered by any table record
    This avoids nb01's prose/table duplication on table pages and makes table records retrievable units.
    """
    out = []
    covered = set()
    table_recs = []
    for rec in tables:
        ps, pe = rec.get("page_start"), rec.get("page_end")
        if isinstance(ps, int) and isinstance(pe, int):
            for p in range(ps, pe + 1):
                covered.add(p)
        ttext = render_table(rec)
        if not ttext.strip():
            continue
        table_recs.append((rec, ttext))
    # table chunks
    for i, (rec, ttext) in enumerate(table_recs):
        ps = rec.get("page_start") if isinstance(rec.get("page_start"), int) else -1
        pe = rec.get("page_end") if isinstance(rec.get("page_end"), int) else ps
        out.append(make_chunk(f"{doc_id}::table_tok512_fixed::table::{i}", ttext, doc_id, ps, pe, "table_tok512_fixed", {"source": "table", "table_logical_key": rec.get("table_logical_key"), "table_extraction_id": rec.get("table_extraction_id"), "statement_kind": rec.get("statement_kind")}))
    # prose chunks only on non-table-covered pages
    for rec in pages:
        page = rec.get("pdf_page_index")
        if page in covered:
            continue
        text = rec.get("page_text") or ""
        spans = tok_spans(text)
        for j, (s, e) in enumerate(windows_by_spans(spans, 512, 0.10)):
            out.append(make_chunk(f"{doc_id}::table_tok512_fixed::p{page}::w{j}", text[s:e], doc_id, page, page, "table_tok512_fixed", {"source": "page_prose"}))
    return out


CHUNK_FUNCS = {
    "C2": lambda doc_id, pages, tables: chunk_c2(doc_id, pages),
    "section_tok512_ov10": lambda doc_id, pages, tables: chunk_section_tok512_ov10(doc_id, pages),
    "table_tok512_fixed": lambda doc_id, pages, tables: chunk_table_tok512_fixed(doc_id, pages, tables),
}


def build_chunks_for_slice(chunker_name, doc_ids):
    fn = CHUNK_FUNCS[chunker_name]
    chunks = []
    for doc_id in doc_ids:
        if doc_id not in DOCS_BY_ID:
            continue
        pages = usable_pages(doc_id)
        tables = load_tables(doc_id)
        chunks.extend(fn(doc_id, pages, tables))
    return chunks


In [7]:
# --- Step 1: table_tok512 bug diagnosis (nb01, read-only analysis) + in-notebook fix ---
# Measured in nb01 (16-doc sample): table_tok512_ov10 -> evidence_containment 0.0, table_records 5655, table_fragmented 0, table_cell_prose_duplicates 19926.
TABLE_BUG_CAUSE = (
    "In nb01, the table strategy chunked page prose with the same token-window path as fixed (512/10%), "
    "so table-region page text was split into arbitrary prose windows, and it also appended a rendered chunk per table record "
    "for the whole document (5655 records) without removing those table pages from prose chunking—creating 19,926 cell-prose duplicates. "
    "Evidence containment only searched prose chunks (ev_chunk_toks) and never the table-rendered chunks, so table evidence could not be "
    "matched via table chunks while prose windows inherited fixed-style boundaries, yielding 0.0 containment alongside heavy duplication. "
    "Fix applied here (table_tok512_fixed): emit rendered table-record chunks and emit prose token windows only for pages not covered by any "
    "table page range, eliminating table-page duplication and making table records first-class retrievable chunks."
)
print(TABLE_BUG_CAUSE)


In nb01, the table strategy chunked page prose with the same token-window path as fixed (512/10%), so table-region page text was split into arbitrary prose windows, and it also appended a rendered chunk per table record for the whole document (5655 records) without removing those table pages from prose chunking—creating 19,926 cell-prose duplicates. Evidence containment only searched prose chunks (ev_chunk_toks) and never the table-rendered chunks, so table evidence could not be matched via table chunks while prose windows inherited fixed-style boundaries, yielding 0.0 containment alongside heavy duplication. Fix applied here (table_tok512_fixed): emit rendered table-record chunks and emit prose token windows only for pages not covered by any table page range, eliminating table-page duplication and making table records first-class retrievable chunks.


In [8]:
# --- build dev eval set and indexed doc slice ---
GOLD_ROWS = load_gold(GOLD_PATH)
EVAL_ROWS = dev_answer_rows_with_evidence(GOLD_ROWS)
SLICE_DOC_IDS = dev_slice_doc_ids(EVAL_ROWS)
EVAL_GOLD_SETS = [gold_evidence_set(r) for r in EVAL_ROWS]
n_ev_records = sum(len(gold_evidence_set(r)) for r in EVAL_ROWS)
n_doc_pages = len(set().union(*EVAL_GOLD_SETS)) if EVAL_GOLD_SETS else 0
print("dev answer rows (eval):", len(EVAL_ROWS))
print("gold evidence (row doc-page unions total):", n_ev_records, "unique doc-pages:", n_doc_pages)
print("indexed dev-evidence doc slice:", len(SLICE_DOC_IDS))


dev answer rows (eval): 45
gold evidence (row doc-page unions total): 55 unique doc-pages: 42
indexed dev-evidence doc slice: 37


In [9]:
# --- retrieval + metrics (function-based) ---

def chunk_matches_gold(chunk_meta, gold_set):
    did = chunk_meta.get("document_id")
    ps = chunk_meta.get("page_start", chunk_meta.get("page"))
    pe = chunk_meta.get("page_end", ps)
    if not isinstance(ps, int):
        return False
    if not isinstance(pe, int):
        pe = ps
    for gd, gp in gold_set:
        if gd == did and ps <= gp <= pe:
            return True
    return False


def bm25_build(chunks):
    corpus_tokens = [tokenize(c.get("text") or "") for c in chunks]
    return BM25Okapi(corpus_tokens)


def bm25_search(bm25, chunks, query, k=10):
    scores = bm25.get_scores(tokenize(query or ""))
    if len(scores) == 0:
        return []
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]


def rrf_fuse(rank_lists, k=60, topn=10):
    # rank_lists: list of lists of (chunk_idx, score) in rank order
    agg = defaultdict(float)
    for rl in rank_lists:
        for rank, (idx, _s) in enumerate(rl, start=1):
            agg[idx] += 1.0 / (k + rank)
    ranked = sorted(agg.items(), key=lambda x: (-x[1], x[0]))
    return [(int(i), float(s)) for i, s in ranked[:topn]]


def eval_retrieval(eval_rows, eval_gold_sets, chunks, search_fn, ks=KS):
    # search_fn(query, k) -> [(chunk_idx, score)]
    n = len(eval_rows)
    recalls = {k: 0 for k in ks}
    mrr_sum = 0.0
    first_ranks = []
    for row, gold in zip(eval_rows, eval_gold_sets):
        q = row.get("question_text") or row.get("question") or ""
        ranked = search_fn(q, max(ks))
        # first relevant rank
        rr = 0.0
        fr = None
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if chunk_matches_gold(chunks[idx]["meta"], gold):
                rr = 1.0 / rank
                fr = rank
                break
        mrr_sum += rr
        first_ranks.append(fr)
        for k in ks:
            hit = any(chunk_matches_gold(chunks[idx]["meta"], gold) for idx, _s in ranked[:k])
            recalls[k] += int(hit)
    return {
        "n": n,
        **{f"recall@{k}": (recalls[k] / n if n else 0.0) for k in ks},
        **{f"hits@{k}": recalls[k] for k in ks},
        "mrr": (mrr_sum / n if n else 0.0),
        "mrr_sum": mrr_sum,
        "first_ranks": first_ranks,
    }


In [10]:
# --- dense embeddings via VM02 (course model) with on-disk cache (function-based) ---
import requests, urllib3


def _safe_text(t, max_chars=3000):
    t = "" if t is None else str(t)
    t = t.replace("\x00", " ")
    t = t if len(t) <= max_chars else t[:max_chars]
    return t if t.strip() else " "


def _post_embeddings(batch_texts, base_url, api_key, model, ca_cert, max_retries=5):
    url = base_url.rstrip("/") + "/embeddings"
    verify = ca_cert if ca_cert and Path(ca_cert).exists() else False
    if verify is False:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {"model": model, "input": batch_texts}
    last_resp = None
    for attempt in range(max_retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=120, verify=verify)
            last_resp = r
            if r.status_code in (429, 503) or r.status_code >= 500:
                time.sleep(min(2 ** attempt, 20))
                continue
            if r.status_code == 422:
                return None, r
            r.raise_for_status()
            data = r.json()
            embs = [d["embedding"] for d in sorted(data.get("data", []), key=lambda x: x.get("index", 0))]
            if len(embs) != len(batch_texts):
                raise RuntimeError(f"embedding count mismatch {len(embs)} vs {len(batch_texts)}")
            return embs, r
        except requests.HTTPError:
            raise
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(min(2 ** attempt, 20))
    return None, last_resp


def _embed_adaptive(batch_texts, base_url, api_key, model, ca_cert, max_retries=5, depth=0):
    embs, resp = _post_embeddings(batch_texts, base_url, api_key, model, ca_cert, max_retries=max_retries)
    if embs is not None:
        return embs
    # 422: split batch or progressively shrink single item
    if len(batch_texts) > 1:
        mid = len(batch_texts) // 2
        left = _embed_adaptive(batch_texts[:mid], base_url, api_key, model, ca_cert, max_retries, depth+1)
        right = _embed_adaptive(batch_texts[mid:], base_url, api_key, model, ca_cert, max_retries, depth+1)
        return left + right
    text = batch_texts[0] or " "
    for max_chars in (1500, 800, 400, 200, 80, 20, 1):
        trial = text[:max_chars]
        if not trial.strip():
            trial = " "
        embs, resp = _post_embeddings([trial], base_url, api_key, model, ca_cert, max_retries=max_retries)
        if embs is not None:
            return embs
    embs, resp = _post_embeddings(["0"], base_url, api_key, model, ca_cert, max_retries=max_retries)
    if embs is not None:
        return embs
    if resp is not None:
        resp.raise_for_status()
    raise RuntimeError("embedding 422 unrecoverable for single item")


def embed_texts_vm02(texts, base_url, api_key, model, ca_cert, batch=16, max_retries=5, label="embed", resume_state=None, checkpoint_fn=None):
    texts = [_safe_text(t, 3000) for t in texts]
    out = [np.asarray(x, dtype=np.float32) for x in (resume_state.tolist() if resume_state is not None else [])]
    start = len(out)
    total = len(texts)
    if start:
        print(f"{label}: resuming from {start}/{total}", flush=True)
    b = 0
    for i in range(start, total, batch):
        b += 1
        chunk = texts[i:i+batch]
        embs = _embed_adaptive(chunk, base_url, api_key, model, ca_cert, max_retries=max_retries)
        out.extend([np.asarray(e, dtype=np.float32) for e in embs])
        done = len(out)
        if b == 1 or b % 10 == 0 or done >= total:
            print(f"{label}: {done}/{total}", flush=True)
        if checkpoint_fn and (b % 5 == 0 or done >= total):
            checkpoint_fn(out)
    arr = np.asarray(out, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return arr / norms


def dense_cache_paths(cache_root, chunker_name):
    base = Path(cache_root) / chunker_name
    return base / "embeddings.npy", base / "chunk_ids.json", base / "meta.json"


def load_or_build_dense(chunks, chunker_name, cache_root=CACHE_ROOT):
    emb_p, ids_p, meta_p = dense_cache_paths(cache_root, chunker_name)
    part_p = emb_p.with_suffix('.partial.npy')
    part_meta_p = emb_p.with_suffix('.partial.meta.json')
    chunk_ids = [c["chunk_id"] for c in chunks]
    text_hash = hashlib.sha256(("\n".join(chunk_ids)).encode("utf-8")).hexdigest()
    want = {"model": EMBED_MODEL, "n": len(chunks), "chunk_ids_sha256": text_hash, "chunker": chunker_name}
    if emb_p.exists() and ids_p.exists() and meta_p.exists():
        try:
            meta = json.loads(meta_p.read_text(encoding="utf-8"))
            ids = json.loads(ids_p.read_text(encoding="utf-8"))
            if meta == want and ids == chunk_ids:
                arr = np.load(emb_p)
                if arr.shape[0] == len(chunks):
                    print(f"dense cache hit: {chunker_name} {arr.shape}", flush=True)
                    for fp in (part_p, part_meta_p):
                        try:
                            if fp.exists():
                                fp.unlink()
                        except Exception:
                            pass
                    return arr, {"cached": True, "shape": list(arr.shape)}
        except Exception:
            pass
    base_url = os.environ.get("LLM_EMBEDDINGS_URL") or os.environ.get("EMBEDDINGS_URL")
    api_key = os.environ.get("EMBEDDINGS_API_KEY") or os.environ.get("LLM_EMBEDDINGS_API_KEY") or ""
    ca_cert = os.environ.get("EMBEDDINGS_CA_CERT") or ""
    if not base_url or not api_key:
        raise RuntimeError("missing embeddings endpoint config in env (LLM_EMBEDDINGS_URL/EMBEDDINGS_API_KEY)")
    texts = [c.get("text") or "" for c in chunks]
    resume_state = None
    if part_p.exists() and part_meta_p.exists():
        try:
            pm = json.loads(part_meta_p.read_text(encoding="utf-8"))
            if pm.get("want") == want:
                arr_p = np.load(part_p)
                if 0 < int(arr_p.shape[0]) <= len(texts):
                    resume_state = arr_p
                    print(f"dense partial found: {chunker_name} {arr_p.shape}", flush=True)
        except Exception:
            resume_state = None
    def ckpt(out_list):
        emb_p.parent.mkdir(parents=True, exist_ok=True)
        np.save(part_p, np.asarray(out_list, dtype=np.float32))
        part_meta_p.write_text(json.dumps({"want": want, "done": len(out_list)}, sort_keys=True), encoding="utf-8")
    print(f"dense cache miss/build: {chunker_name} n_chunks={len(chunks)} via {base_url}", flush=True)
    t0 = time.time()
    arr = embed_texts_vm02(texts, base_url, api_key, EMBED_MODEL, ca_cert, batch=EMBED_BATCH, max_retries=EMBED_MAX_RETRIES, label=f"doc_emb:{chunker_name}", resume_state=resume_state, checkpoint_fn=ckpt)
    sec = time.time() - t0
    emb_p.parent.mkdir(parents=True, exist_ok=True)
    np.save(emb_p, arr)
    ids_p.write_text(json.dumps(chunk_ids), encoding="utf-8")
    meta_p.write_text(json.dumps(want, sort_keys=True), encoding="utf-8")
    for fp in (part_p, part_meta_p):
        try:
            if fp.exists():
                fp.unlink()
        except Exception:
            pass
    print(f"dense built+cached: {chunker_name} shape={arr.shape} sec={sec:.2f}", flush=True)
    return arr, {"cached": False, "shape": list(arr.shape), "build_seconds": round(sec, 3)}


def dense_search(embs, query_vec, k=10):
    scores = embs @ query_vec
    k = min(k, len(scores))
    if k <= 0:
        return []
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]



In [11]:
# --- run all chunkers x retrievers (sequential) ---
MATCHING_RULE = (
    "Chunk-level doc/page match (not full-excerpt containment): for a query, gold relevant G is the set of "
    "(document_id, pdf_page) from its gold evidence records. A retrieved chunk is a hit if chunk.document_id equals a gold "
    "document_id and the gold pdf_page lies within [chunk.page_start, chunk.page_end] (inclusive). "
    "recall@k = fraction of eval rows with >=1 hit in top-k; MRR = mean(1/rank of first hit) over rows (0 if none). "
    "Hybrid = Reciprocal Rank Fusion over BM25 and dense rankings with k=60: score(d)=sum 1/(60+rank)."
)

def metrics_from_rankings(rankings, eval_rows, eval_gold_sets, chunks, ks=KS):
    n = len(eval_rows)
    recalls = {k: 0 for k in ks}
    mrr_sum = 0.0
    first_ranks = []
    for qi, (row, gold) in enumerate(zip(eval_rows, eval_gold_sets)):
        ranked = rankings[qi]
        rr = 0.0
        fr = None
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if chunk_matches_gold(chunks[idx]["meta"], gold):
                rr = 1.0 / rank
                fr = rank
                break
        mrr_sum += rr
        first_ranks.append(fr)
        for k in ks:
            hit = any(chunk_matches_gold(chunks[idx]["meta"], gold) for idx, _s in ranked[:k])
            recalls[k] += int(hit)
    return {
        "n": n,
        **{f"recall@{k}": (recalls[k] / n if n else 0.0) for k in ks},
        **{f"hits@{k}": recalls[k] for k in ks},
        "mrr": (mrr_sum / n if n else 0.0),
        "mrr_sum": mrr_sum,
        "first_ranks": first_ranks,
    }

all_results = {"chunkers": {}, "matching_rule": MATCHING_RULE, "rrf_k": RRF_K, "ks": list(KS)}
chunk_summaries = {}

# Pre-embed eval queries once (reused across all chunkers/retrievers) to avoid repeated network calls.
base_url = os.environ.get("LLM_EMBEDDINGS_URL") or os.environ.get("EMBEDDINGS_URL")
api_key = os.environ.get("EMBEDDINGS_API_KEY") or os.environ.get("LLM_EMBEDDINGS_API_KEY") or ""
ca_cert = os.environ.get("EMBEDDINGS_CA_CERT") or ""
EVAL_QUERIES = [(r.get("question_text") or r.get("question") or "") for r in EVAL_ROWS]
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
q_cache = Path(CACHE_ROOT) / "eval_query_embeddings.npy"
q_meta = Path(CACHE_ROOT) / "eval_query_embeddings.meta.json"
q_want = {"model": EMBED_MODEL, "n": len(EVAL_QUERIES), "gold_path": GOLD_PATH}
if q_cache.exists() and q_meta.exists() and json.loads(q_meta.read_text()).get("model")==EMBED_MODEL and json.loads(q_meta.read_text()).get("n")==len(EVAL_QUERIES):
    QUERY_EMBS = np.load(q_cache)
    print(f"query embeddings cache hit: {QUERY_EMBS.shape}", flush=True)
else:
    print(f"embedding eval queries: {len(EVAL_QUERIES)}", flush=True)
    QUERY_EMBS = embed_texts_vm02(EVAL_QUERIES, base_url, api_key, EMBED_MODEL, ca_cert, batch=EMBED_BATCH, max_retries=EMBED_MAX_RETRIES, label="query_emb")
    np.save(q_cache, QUERY_EMBS)
    q_meta.write_text(json.dumps(q_want, sort_keys=True), encoding="utf-8")
    print(f"query embeddings cached: {QUERY_EMBS.shape}", flush=True)

for cname in CHUNKERS:
    print(f"\n=== CHUNKER {cname} ===", flush=True)
    t0 = time.time()
    chunks = build_chunks_for_slice(cname, SLICE_DOC_IDS)
    t_chunk = time.time() - t0
    print(f"chunks: {len(chunks)} built in {t_chunk:.2f}s", flush=True)
    # BM25
    t1 = time.time()
    bm25 = bm25_build(chunks)
    t_bm25_build = time.time() - t1
    bm25_rankings = [bm25_search(bm25, chunks, q, max(KS)) for q in EVAL_QUERIES]
    bm25_eval = metrics_from_rankings(bm25_rankings, EVAL_ROWS, EVAL_GOLD_SETS, chunks)
    # Dense (doc embeddings cached to disk; query embeddings precomputed above)
    embs, emb_info = load_or_build_dense(chunks, cname, CACHE_ROOT)
    dense_rankings = [dense_search(embs, QUERY_EMBS[qi], max(KS)) for qi in range(len(EVAL_QUERIES))]
    dense_eval = metrics_from_rankings(dense_rankings, EVAL_ROWS, EVAL_GOLD_SETS, chunks)
    # Hybrid RRF (reuses per-query rankings, no extra embedding/search calls)
    hybrid_rankings = [rrf_fuse([bm25_rankings[qi], dense_rankings[qi]], k=RRF_K, topn=max(KS)) for qi in range(len(EVAL_QUERIES))]
    hybrid_eval = metrics_from_rankings(hybrid_rankings, EVAL_ROWS, EVAL_GOLD_SETS, chunks)

    all_results["chunkers"][cname] = {
        "n_chunks": len(chunks),
        "chunk_build_seconds": round(t_chunk, 3),
        "bm25_build_seconds": round(t_bm25_build, 3),
        "dense_info": emb_info,
        "bm25": bm25_eval,
        "dense": dense_eval,
        "hybrid_rrf": hybrid_eval,
    }
    chunk_summaries[cname] = len(chunks)
    for mname, mev in [("bm25", bm25_eval), ("dense", dense_eval), ("hybrid_rrf", hybrid_eval)]:
        print(mname, {f"r@{k}": round(mev[f'recall@{k}'],4) for k in KS}, "MRR", round(mev["mrr"],4), flush=True)

print("\nchunk counts:", chunk_summaries, flush=True)



query embeddings cache hit: (45, 384)

=== CHUNKER C2 ===
chunks: 50221 built in 98.35s
dense cache hit: C2 (50221, 384)
bm25 {'r@1': 0.0, 'r@3': 0.0, 'r@5': 0.0, 'r@10': 0.0222} MRR 0.0037
dense {'r@1': 0.0667, 'r@3': 0.1333, 'r@5': 0.1556, 'r@10': 0.1556} MRR 0.1007
hybrid_rrf {'r@1': 0.0, 'r@3': 0.0889, 'r@5': 0.1333, 'r@10': 0.1556} MRR 0.0543

=== CHUNKER section_tok512_ov10 ===
chunks: 102265 built in 75.35s
dense cache hit: section_tok512_ov10 (102265, 384)
bm25 {'r@1': 0.0, 'r@3': 0.0222, 'r@5': 0.0222, 'r@10': 0.0444} MRR 0.0106
dense {'r@1': 0.0, 'r@3': 0.0222, 'r@5': 0.0667, 'r@10': 0.0667} MRR 0.0222
hybrid_rrf {'r@1': 0.0, 'r@3': 0.0, 'r@5': 0.0444, 'r@10': 0.0889} MRR 0.0163

=== CHUNKER table_tok512_fixed ===
chunks: 23091 built in 17.50s
dense cache hit: table_tok512_fixed (23091, 384)
bm25 {'r@1': 0.0222, 'r@3': 0.0222, 'r@5': 0.0222, 'r@10': 0.0222} MRR 0.0222
dense {'r@1': 0.0222, 'r@3': 0.0222, 'r@5': 0.0222, 'r@10': 0.0222} MRR 0.0222
hybrid_rrf {'r@1': 0.0222, 'r@

In [12]:
# --- save results JSON and markdown report ---
report = {
    "experiment": "week2_retrieval_recall",
    "status": "complete",
    "corpus_manifest_sha256": FREEZE_SHA,
    "tokenizer_version": TOKENIZER_VERSION,
    "gold_path": GOLD_PATH,
    "eval_split": "dev",
    "eval_expected_behaviour": "answer",
    "n_eval_rows": len(EVAL_ROWS),
    "n_eval_unique_doc_pages": n_doc_pages,
    "indexed_slice": "dev_answer_evidence_document_ids",
    "indexed_doc_ids": SLICE_DOC_IDS,
    "n_indexed_docs": len(SLICE_DOC_IDS),
    "chunkers": list(CHUNKERS),
    "embedding_model": EMBED_MODEL,
    "rrf_k": RRF_K,
    "ks": list(KS),
    "matching_rule": MATCHING_RULE,
    "table_bug_cause": TABLE_BUG_CAUSE,
    "results": all_results["chunkers"],
}
Path(RESULTS_JSON).parent.mkdir(parents=True, exist_ok=True)
Path(RESULTS_JSON).write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
print("wrote", RESULTS_JSON)

# markdown
lines = []
lines.append("# Week 2 Retrieval Recall Analysis\n")
lines.append("Read-only on frozen assets, notebook 01, and existing JSON. No winner claims.\n")
lines.append("## Setup (counts)\n")
lines.append(f"- Corpus manifest sha256: `{FREEZE_SHA}`")
lines.append(f"- Gold: `{GOLD_PATH}` (dev, expected_behaviour=answer)")
lines.append(f"- Eval rows: **{len(EVAL_ROWS)}**, unique gold doc-pages: **{n_doc_pages}**, indexed dev-evidence docs: **{len(SLICE_DOC_IDS)}**")
lines.append(f"- Chunkers: {', '.join(CHUNKERS)} | Dense model: `{EMBED_MODEL}` (VM02, cached to disk) | RRF k={RRF_K} | k={list(KS)}")
lines.append("\n## Matching rule\n")
lines.append(MATCHING_RULE + "\n")
lines.append("## Step 1 — table_tok512 bug (nb01, read-only) and fix (this notebook)\n")
lines.append(TABLE_BUG_CAUSE + "\n")
lines.append("## Results\n")
header = "| Chunker | Retriever | n_chunks | recall@1 | recall@3 | recall@5 | recall@10 | MRR | hits@1 | hits@3 | hits@5 | hits@10 | n |"
sep = "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"
lines.append(header); lines.append(sep)
for cname in CHUNKERS:
    r = all_results["chunkers"][cname]
    for mname, label in [("bm25","BM25"),("dense","Dense"),("hybrid_rrf","Hybrid RRF (k=60)")]:
        m = r[mname]
        lines.append(f"| {cname} | {label} | {r['n_chunks']} | {m['recall@1']:.4f} | {m['recall@3']:.4f} | {m['recall@5']:.4f} | {m['recall@10']:.4f} | {m['mrr']:.4f} | {m['hits@1']} | {m['hits@3']} | {m['hits@5']} | {m['hits@10']} | {m['n']} |")
lines.append("\n*Counts only. No winner selected.*\n")
Path(ANALYSIS_MD).parent.mkdir(parents=True, exist_ok=True)
Path(ANALYSIS_MD).write_text("\n".join(lines), encoding="utf-8")
print("wrote", ANALYSIS_MD)


wrote results/week2_retrieval.json
wrote docs/WEEK2_RETRIEVAL_ANALYSIS.md


## Diagnosis (C2 only, cached embeddings, no re-embed)
Read-only diagnosis added at end. Reuses `results/week2_retrieval_cache/C2/*` and `eval_query_embeddings.npy`; does not re-embed and does not modify existing matching rule or result files.


In [13]:
# --- Diagnosis setup: C2 only, reuse cached embeddings (no network/re-embed) ---
import numpy as np
from pathlib import Path

assert 'build_chunks_for_slice' in globals(), "run earlier notebook cells first (helpers required)"
assert 'SLICE_DOC_IDS' in globals() and 'EVAL_ROWS' in globals() and 'EVAL_GOLD_SETS' in globals(), "run eval-slice cell first"
assert 'EVAL_QUERIES' in globals(), "run retrieval cell or define EVAL_QUERIES"
assert 'CACHE_ROOT' in globals() and 'KS' in globals() and 'RRF_K' in globals()

cname = "C2"
chunks_c2 = build_chunks_for_slice("C2", SLICE_DOC_IDS)
cache_dir = Path(CACHE_ROOT) / "C2"
embs_c2 = np.load(cache_dir / "embeddings.npy")
cache_ids = json.loads((cache_dir / "chunk_ids.json").read_text(encoding="utf-8"))
qembs = np.load(Path(CACHE_ROOT) / "eval_query_embeddings.npy")

chunk_ids_now = [c["chunk_id"] for c in chunks_c2]
print("C2 chunks:", len(chunks_c2))
print("cached doc embeddings:", embs_c2.shape, "| cached chunk_ids match:", cache_ids == chunk_ids_now)
print("cached query embeddings:", qembs.shape, "| n_eval:", len(EVAL_ROWS))
assert cache_ids == chunk_ids_now, "C2 chunk_ids do not match cached embeddings"
assert qembs.shape[0] == len(EVAL_ROWS) == len(EVAL_QUERIES), "query embedding / eval row count mismatch"

# rankings (local only)
bm25_c2 = bm25_build(chunks_c2)
bm25_rank_c2 = [bm25_search(bm25_c2, chunks_c2, q, max(KS)) for q in EVAL_QUERIES]
dense_rank_c2 = [dense_search(embs_c2, qembs[i], max(KS)) for i in range(len(EVAL_QUERIES))]
hybrid_rank_c2 = [rrf_fuse([bm25_rank_c2[i], dense_rank_c2[i]], k=RRF_K, topn=max(KS)) for i in range(len(EVAL_QUERIES))]
rank_sets = {"BM25": bm25_rank_c2, "Dense": dense_rank_c2, "Hybrid_RRF": hybrid_rank_c2}
print("rankings ready (BM25/Dense/Hybrid), no re-embed")


C2 chunks: 50221
cached doc embeddings: (50221, 384) | cached chunk_ids match: True
cached query embeddings: (45, 384) | n_eval: 45
rankings ready (BM25/Dense/Hybrid), no re-embed


In [14]:
# --- 1) Doc-level vs page-level recall (C2) ---
def chunk_matches_gold_doc(chunk_meta, gold_set):
    did = chunk_meta.get("document_id")
    return any(gd == did for gd, _gp in gold_set)

def score_with_matcher(rankings, chunks, matcher, ks=KS):
    n = len(EVAL_ROWS)
    recalls = {k: 0 for k in ks}
    mrr_sum = 0.0
    for qi, gold in enumerate(EVAL_GOLD_SETS):
        ranked = rankings[qi]
        rr = 0.0
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if matcher(chunks[idx]["meta"], gold):
                rr = 1.0 / rank
                break
        mrr_sum += rr
        for k in ks:
            if any(matcher(chunks[idx]["meta"], gold) for idx, _s in ranked[:k]):
                recalls[k] += 1
    return {"n": n, **{f"recall@{k}": recalls[k]/n for k in ks}, **{f"hits@{k}": recalls[k] for k in ks}, "mrr": mrr_sum/n}

print("=== 1. Doc-level recall (hit if chunk.document_id in gold doc ids) vs page-level (offset 0) ===")
for name, ranks in rank_sets.items():
    doc_m = score_with_matcher(ranks, chunks_c2, chunk_matches_gold_doc)
    page_m = score_with_matcher(ranks, chunks_c2, lambda meta, gold: chunk_matches_gold(meta, gold))
    print(f"\n[{name}] n={doc_m['n']}")
    print("  doc-level :", {f"R@{k}": round(doc_m[f'recall@{k}'],4) for k in KS}, "MRR", round(doc_m['mrr'],4), "hits", {k: doc_m[f'hits@{k}'] for k in KS})
    print("  page-level:", {f"R@{k}": round(page_m[f'recall@{k}'],4) for k in KS}, "MRR", round(page_m['mrr'],4), "hits", {k: page_m[f'hits@{k}'] for k in KS})


=== 1. Doc-level recall (hit if chunk.document_id in gold doc ids) vs page-level (offset 0) ===

[BM25] n=45
  doc-level : {'R@1': 0.2667, 'R@3': 0.4667, 'R@5': 0.5778, 'R@10': 0.7111} MRR 0.3961 hits {1: 12, 3: 21, 5: 26, 10: 32}
  page-level: {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0222} MRR 0.0037 hits {1: 0, 3: 0, 5: 0, 10: 1}

[Dense] n=45
  doc-level : {'R@1': 0.6, 'R@3': 0.7333, 'R@5': 0.7556, 'R@10': 0.8222} MRR 0.6784 hits {1: 27, 3: 33, 5: 34, 10: 37}
  page-level: {'R@1': 0.0667, 'R@3': 0.1333, 'R@5': 0.1556, 'R@10': 0.1556} MRR 0.1007 hits {1: 3, 3: 6, 5: 7, 10: 7}

[Hybrid_RRF] n=45
  doc-level : {'R@1': 0.4889, 'R@3': 0.7333, 'R@5': 0.8444, 'R@10': 0.8667} MRR 0.6378 hits {1: 22, 3: 33, 5: 38, 10: 39}
  page-level: {'R@1': 0.0, 'R@3': 0.0889, 'R@5': 0.1333, 'R@10': 0.1556} MRR 0.0543 hits {1: 0, 3: 4, 5: 6, 10: 7}


In [15]:
# --- 2) Page offset test: gold pdf_page shifted by -1, 0, +1 ---
def chunk_matches_gold_shift(chunk_meta, gold_set, shift=0):
    did = chunk_meta.get("document_id")
    ps = chunk_meta.get("page_start", chunk_meta.get("page"))
    pe = chunk_meta.get("page_end", ps)
    if not isinstance(ps, int):
        return False
    if not isinstance(pe, int):
        pe = ps
    for gd, gp in gold_set:
        if gd == did and ps <= (gp + shift) <= pe:
            return True
    return False

print("=== 2. Page offset test (page-level recall, shifts -1/0/+1 on gold pdf_page) ===")
for shift in (-1, 0, 1):
    print(f"\n-- shift {shift:+d} --")
    for name, ranks in rank_sets.items():
        m = score_with_matcher(ranks, chunks_c2, lambda meta, gold, s=shift: chunk_matches_gold_shift(meta, gold, s))
        print(f"  {name:10s}", {f"R@{k}": round(m[f'recall@{k}'],4) for k in KS}, "MRR", round(m['mrr'],4), "hits", {k: m[f'hits@{k}'] for k in KS})


=== 2. Page offset test (page-level recall, shifts -1/0/+1 on gold pdf_page) ===

-- shift -1 --
  BM25       {'R@1': 0.0222, 'R@3': 0.0444, 'R@5': 0.0444, 'R@10': 0.0667} MRR 0.0321 hits {1: 1, 3: 2, 5: 2, 10: 3}
  Dense      {'R@1': 0.0, 'R@3': 0.0889, 'R@5': 0.1333, 'R@10': 0.1556} MRR 0.0484 hits {1: 0, 3: 4, 5: 6, 10: 7}
  Hybrid_RRF {'R@1': 0.0, 'R@3': 0.0444, 'R@5': 0.0889, 'R@10': 0.1556} MRR 0.0381 hits {1: 0, 3: 2, 5: 4, 10: 7}

-- shift +0 --
  BM25       {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0222} MRR 0.0037 hits {1: 0, 3: 0, 5: 0, 10: 1}
  Dense      {'R@1': 0.0667, 'R@3': 0.1333, 'R@5': 0.1556, 'R@10': 0.1556} MRR 0.1007 hits {1: 3, 3: 6, 5: 7, 10: 7}
  Hybrid_RRF {'R@1': 0.0, 'R@3': 0.0889, 'R@5': 0.1333, 'R@10': 0.1556} MRR 0.0543 hits {1: 0, 3: 4, 5: 6, 10: 7}

-- shift +1 --
  BM25       {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0444} MRR 0.0069 hits {1: 0, 3: 0, 5: 0, 10: 2}
  Dense      {'R@1': 0.0222, 'R@3': 0.0444, 'R@5': 0.0889, 'R@10': 0.1556} MRR 

In [16]:
# --- 3) Miss analysis: 5 eval rows missing in Hybrid RRF page-level @10 (offset 0) ---
def first_hit_rank(rankings, qi, gold, chunks, matcher):
    for rank, (idx, _s) in enumerate(rankings[qi], start=1):
        if matcher(chunks[idx]["meta"], gold):
            return rank
    return None

miss_qis = []
for qi, gold in enumerate(EVAL_GOLD_SETS):
    fr = first_hit_rank(hybrid_rank_c2, qi, gold, chunks_c2, lambda m,g: chunk_matches_gold(m,g))
    if fr is None or fr > 10:
        miss_qis.append((qi, fr))
print("=== 3. Misses: Hybrid RRF page-level @10 (offset 0) ===")
print("total misses @10:", len(miss_qis))
show = miss_qis[:5]
for n, (qi, fr) in enumerate(show, 1):
    row = EVAL_ROWS[qi]
    q = row.get("question_text") or row.get("question") or ""
    print(f"\n--- Miss {n} (eval idx {qi}, first_hit_rank={fr}) ---")
    print("QUERY:", q)
    evs = [e for e in (row.get("evidence") or []) if isinstance(e, dict)]
    if not evs:
        print("  (no evidence entries)")
    for ei, e in enumerate(evs, 1):
        did = e.get("document_id")
        pg = e.get("pdf_page")
        excerpt = e.get("excerpt") or ""
        print(f"  evidence[{ei}] doc_id={did} pdf_page={pg}")
        print(f"  excerpt[:200]: {excerpt[:200]!r}")
        # excerpt presence in any C2 chunk of that doc
        exact_hits = []
        norm_hits = []
        ex_norm = " ".join(excerpt.split())
        for c in chunks_c2:
            if c["meta"].get("document_id") != did:
                continue
            txt = c.get("text") or ""
            if excerpt and excerpt in txt:
                exact_hits.append((c["meta"].get("page_start"), c["meta"].get("page_end"), c["chunk_id"]))
            elif ex_norm and ex_norm in " ".join(txt.split()):
                norm_hits.append((c["meta"].get("page_start"), c["meta"].get("page_end"), c["chunk_id"]))
        if exact_hits:
            print(f"  excerpt_found_exact_in_doc_chunks: YES count={len(exact_hits)} pages/ranges(sample<=5)={[(a,b) for a,b,_ in exact_hits[:5]]}")
        else:
            print("  excerpt_found_exact_in_doc_chunks: NO")
        if (not exact_hits) and norm_hits:
            print(f"  excerpt_found_ws_norm_in_doc_chunks: YES count={len(norm_hits)} pages/ranges(sample<=5)={[(a,b) for a,b,_ in norm_hits[:5]]}")
        elif not exact_hits:
            print("  excerpt_found_ws_norm_in_doc_chunks: NO")


=== 3. Misses: Hybrid RRF page-level @10 (offset 0) ===
total misses @10: 38

--- Miss 1 (eval idx 0, first_hit_rank=None) ---
QUERY: Between Deepak Nitrite and Srf, on total assets for the year ended 31 March 2025 consolidated results, which prints the larger figure, and what are the two figures (in Rs. in crore, as printed on each statement)?
  evidence[1] doc_id=fa7dac2401f10d824c6e6b3e5dd482a0e0ab09b50e6164eeebc289ec952fff31 pdf_page=363
  excerpt[:200]: 'TOTAL ASSETS 7,717.67 6,096.22'
  excerpt_found_exact_in_doc_chunks: YES count=1 pages/ranges(sample<=5)=[(363, 363)]
  evidence[2] doc_id=8e4e186331c2ece16878e3f3a29e1c92cd24c2fa1192e634a37e29593f706862 pdf_page=177
  excerpt[:200]: 'TOTAL ASSETS 21,557.12 20,481.94'
  excerpt_found_exact_in_doc_chunks: YES count=1 pages/ranges(sample<=5)=[(177, 177)]

--- Miss 2 (eval idx 1, first_hit_rank=None) ---
QUERY: For the year ended 31 March 2025, consolidated (both companies): is total liabilities larger for Deepak Nitrite or for Srf, 

In [17]:
# --- 4) First 3 queries and their gold excerpts ---
print("=== 4. First 3 eval queries and gold excerpts ===")
for qi in range(min(3, len(EVAL_ROWS))):
    row = EVAL_ROWS[qi]
    q = row.get("question_text") or row.get("question") or ""
    print(f"\n[Query {qi+1} | eval idx {qi} | row_id={row.get('row_id') or row.get('id')}]")
    print("Q:", q)
    evs = [e for e in (row.get("evidence") or []) if isinstance(e, dict)]
    if not evs:
        print("  (no evidence)")
    for ei, e in enumerate(evs, 1):
        did = e.get("document_id")
        pg = e.get("pdf_page")
        excerpt = e.get("excerpt") or ""
        role = e.get("evidence_role")
        print(f"  evidence[{ei}] role={role} doc_id={did} pdf_page={pg}")
        print(f"  excerpt: {excerpt!r}")
print("\n--- diagnosis end (no files written, no matching rule changed) ---")


=== 4. First 3 eval queries and gold excerpts ===

[Query 1 | eval idx 0 | row_id=G-001]
Q: Between Deepak Nitrite and Srf, on total assets for the year ended 31 March 2025 consolidated results, which prints the larger figure, and what are the two figures (in Rs. in crore, as printed on each statement)?
  evidence[1] role=primary doc_id=fa7dac2401f10d824c6e6b3e5dd482a0e0ab09b50e6164eeebc289ec952fff31 pdf_page=363
  excerpt: 'TOTAL ASSETS 7,717.67 6,096.22'
  evidence[2] role=comparison doc_id=8e4e186331c2ece16878e3f3a29e1c92cd24c2fa1192e634a37e29593f706862 pdf_page=177
  excerpt: 'TOTAL ASSETS 21,557.12 20,481.94'

[Query 2 | eval idx 1 | row_id=G-003]
Q: For the year ended 31 March 2025, consolidated (both companies): is total liabilities larger for Deepak Nitrite or for Srf, and what are the two figures in Rs. in crore as printed?
  evidence[1] role=primary doc_id=fa7dac2401f10d824c6e6b3e5dd482a0e0ab09b50e6164eeebc289ec952fff31 pdf_page=363
  excerpt: 'Total Liabilities 2,293.01 1,27

## Extended diagnosis (C2, cached, no re-embed)
Appended diagnosis tasks 1–5. C2 only, reused cached embeddings/query embeddings; no re-embedding; no changes to existing matching rules or result files.


In [18]:
# --- Extended setup: C2 only, reuse cached embeddings/rankings if available ---
import re, numpy as np
from pathlib import Path

assert 'build_chunks_for_slice' in globals(), "run earlier notebook cells first (helpers required)"
assert 'SLICE_DOC_IDS' in globals() and 'EVAL_ROWS' in globals() and 'EVAL_GOLD_SETS' in globals(), "run eval-slice cell first"
assert 'CACHE_ROOT' in globals() and 'KS' in globals() and 'RRF_K' in globals()
assert 'bm25_build' in globals() and 'bm25_search' in globals() and 'dense_search' in globals() and 'rrf_fuse' in globals()
assert 'chunk_matches_gold' in globals() and 'TOKENIZER_VERSION' in globals()

EVAL_QUERIES = [(r.get("question_text") or r.get("question") or "") for r in EVAL_ROWS]

def _ensure_c2_state():
    global chunks_c2, embs_c2, qembs, cache_ids, chunk_ids_now
    global bm25_c2, bm25_rank_c2, dense_rank_c2, hybrid_rank_c2, rank_sets
    if 'chunks_c2' not in globals():
        chunks_c2 = build_chunks_for_slice("C2", SLICE_DOC_IDS)
    cache_dir = Path(CACHE_ROOT) / "C2"
    embs_c2 = np.load(cache_dir / "embeddings.npy")
    cache_ids = json.loads((cache_dir / "chunk_ids.json").read_text(encoding="utf-8"))
    qembs = np.load(Path(CACHE_ROOT) / "eval_query_embeddings.npy")
    chunk_ids_now = [c["chunk_id"] for c in chunks_c2]
    assert cache_ids == chunk_ids_now, "C2 chunk_ids do not match cached embeddings"
    assert qembs.shape[0] == len(EVAL_ROWS) == len(EVAL_QUERIES)
    if 'bm25_c2' not in globals():
        bm25_c2 = bm25_build(chunks_c2)
    if 'bm25_rank_c2' not in globals():
        bm25_rank_c2 = [bm25_search(bm25_c2, chunks_c2, q, max(KS)) for q in EVAL_QUERIES]
    if 'dense_rank_c2' not in globals():
        dense_rank_c2 = [dense_search(embs_c2, qembs[i], max(KS)) for i in range(len(EVAL_QUERIES))]
    if 'hybrid_rank_c2' not in globals():
        hybrid_rank_c2 = [rrf_fuse([bm25_rank_c2[i], dense_rank_c2[i]], k=RRF_K, topn=max(KS)) for i in range(len(EVAL_QUERIES))]
    rank_sets = {"BM25": bm25_rank_c2, "Dense": dense_rank_c2, "Hybrid_RRF": hybrid_rank_c2}

_ensure_c2_state()

def score_page_level(rankings, chunks, ks=KS):
    n = len(EVAL_ROWS); recalls = {k: 0 for k in ks}; mrr_sum = 0.0
    for qi, gold in enumerate(EVAL_GOLD_SETS):
        ranked = rankings[qi]; rr = 0.0
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if chunk_matches_gold(chunks[idx]["meta"], gold):
                rr = 1.0 / rank; break
        mrr_sum += rr
        for k in ks:
            if any(chunk_matches_gold(chunks[idx]["meta"], gold) for idx, _s in ranked[:k]):
                recalls[k] += 1
    return {"n": n, **{f"recall@{k}": recalls[k]/n for k in ks}, **{f"hits@{k}": recalls[k] for k in ks}, "mrr": mrr_sum/n}

def _fmt(m, ks=KS):
    return {f"R@{k}": round(m[f'recall@{k}'], 4) for k in ks} | {"MRR": round(m['mrr'], 4), "hits": {k: m[f'hits@{k}'] for k in ks}}

print("extended setup ready | C2 chunks", len(chunks_c2), "emb", embs_c2.shape, "qemb", qembs.shape, "eval_n", len(EVAL_ROWS))


extended setup ready | C2 chunks 50221 emb (50221, 384) qemb (45, 384) eval_n 45


In [19]:
# --- 1) BM25 tokenization check: before vs after (lowercase + punctuation stripped + stopword removal) ---
from rank_bm25 import BM25Okapi
from src.rag_ingestion.tokenizer import tokenize as tokenizer_v1, TOKENIZER_VERSION

STOPWORDS_SIMPLE = set("""
a an and are as at be by for from has have he in is it its of on or that the to was were will with this these those but if into no not than then so such can may we our their they you your i me my
""".split())

def simple_tok(text):
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    toks = text.split()
    return [t for t in toks if t not in STOPWORDS_SIMPLE and len(t) > 1]

print("=== 1. BM25 tokenization check (C2, page-level) ===")
print("tokenizer used (before):", f"src.rag_ingestion.tokenizer.tokenize ({TOKENIZER_VERSION})", "| numeric-preserving, NFC+lower, [a-z0-9]+ with FY alternates")
print("sample(v1):", tokenizer_v1("TOTAL ASSETS 7,717.67 FY2025 (1,234)"))
print("tokenizer (after): lowercase + punctuation stripped + stopword removal | stopwords_n=", len(STOPWORDS_SIMPLE))
print("sample(after):", simple_tok("TOTAL ASSETS 7,717.67 FY2025 (1,234)"))

# before (cached rankings exist)
bm25_before = score_page_level(bm25_rank_c2, chunks_c2, ks=KS)
print("\nBEFORE (tokenizer.v1):", _fmt(bm25_before))

# after: rebuild BM25 with simple tokenizer (CPU only, no re-embed)
corpus_simple = [simple_tok(c.get("text") or "") for c in chunks_c2]
bm25_simple = BM25Okapi(corpus_simple)
bm25_rank_simple = []
for q in EVAL_QUERIES:
    scores = bm25_simple.get_scores(simple_tok(q))
    k = min(max(KS), len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    bm25_rank_simple.append([(int(i), float(scores[i])) for i in idx])
bm25_after = score_page_level(bm25_rank_simple, chunks_c2, ks=KS)
print("AFTER  (lower+punct-strip+stopwords):", _fmt(bm25_after))


=== 1. BM25 tokenization check (C2, page-level) ===
tokenizer used (before): src.rag_ingestion.tokenizer.tokenize (tokenizer.v1) | numeric-preserving, NFC+lower, [a-z0-9]+ with FY alternates
sample(v1): ['total', 'assets', '7717.67', 'fy2025', '-1234', '2025']
tokenizer (after): lowercase + punctuation stripped + stopword removal | stopwords_n= 50
sample(after): ['total', 'assets', '717', '67', 'fy2025', '234']

BEFORE (tokenizer.v1): {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0222, 'MRR': 0.0037, 'hits': {1: 0, 3: 0, 5: 0, 10: 1}}
AFTER  (lower+punct-strip+stopwords): {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.1111, 'MRR': 0.0153, 'hits': {1: 0, 3: 0, 5: 0, 10: 5}}


In [20]:
# --- 2) Candidate-pool depth: page-level recall@20/50/100 (C2) ---
DEPTH_KS = (20, 50, 100)

def topk_from_scores(scores, k):
    k = min(k, len(scores))
    if k <= 0:
        return []
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]

bm25_scores_all = [bm25_c2.get_scores(tokenize(q or "")) for q in EVAL_QUERIES]
dense_scores_all = [embs_c2 @ qembs[i] for i in range(len(EVAL_QUERIES))]

def ranks_at_k(k):
    b = [topk_from_scores(bm25_scores_all[i], k) for i in range(len(EVAL_QUERIES))]
    d = [topk_from_scores(dense_scores_all[i], k) for i in range(len(EVAL_QUERIES))]
    h = [rrf_fuse([b[i], d[i]], k=RRF_K, topn=k) for i in range(len(EVAL_QUERIES))]
    return {"BM25": b, "Dense": d, "Hybrid_RRF": h}

print("=== 2. Candidate-pool depth (page-level recall@20/50/100, C2) ===")
for k in DEPTH_KS:
    rs = ranks_at_k(k)
    print(f"\n-- @{k} --")
    for name in ("BM25", "Dense", "Hybrid_RRF"):
        m = score_page_level(rs[name], chunks_c2, ks=(k,))
        print(f"  {name:10s} R@{k}={round(m[f'recall@{k}'],4)} hits@{k}={m[f'hits@{k}']}/{m['n']} MRR={round(m['mrr'],4)}")


=== 2. Candidate-pool depth (page-level recall@20/50/100, C2) ===

-- @20 --
  BM25       R@20=0.1333 hits@20=6/45 MRR=0.0114
  Dense      R@20=0.2 hits@20=9/45 MRR=0.1041
  Hybrid_RRF R@20=0.1556 hits@20=7/45 MRR=0.0588

-- @50 --
  BM25       R@50=0.2667 hits@50=12/45 MRR=0.0151
  Dense      R@50=0.4 hits@50=18/45 MRR=0.1105
  Hybrid_RRF R@50=0.3778 hits@50=17/45 MRR=0.0568

-- @100 --
  BM25       R@100=0.3778 hits@100=17/45 MRR=0.0168
  Dense      R@100=0.5333 hits@100=24/45 MRR=0.1124
  Hybrid_RRF R@100=0.6 hits@100=27/45 MRR=0.0635


In [21]:
# --- 3) Evidence-level coverage @10 (fraction of gold doc,page pairs hit in top-10, avg over rows) + any-hit recall ---
def evidence_coverage(rankings, chunks, k=10):
    per_row = []
    for qi, gold in enumerate(EVAL_GOLD_SETS):
        ranked = rankings[qi][:k]
        if not gold:
            per_row.append(0.0); continue
        hit_pairs = 0
        for g in gold:
            if any(chunk_matches_gold(chunks[idx]["meta"], {g}) for idx, _s in ranked):
                hit_pairs += 1
        per_row.append(hit_pairs / len(gold))
    anyhit = score_page_level(rankings, chunks, ks=(k,))
    return {"k": k, "n": len(per_row), "avg_pair_fraction_hit": sum(per_row)/len(per_row) if per_row else 0.0, "any_hit_recall": anyhit[f"recall@{k}"], "any_hit_hits": anyhit[f"hits@{k}"]}

print("=== 3. Evidence-level coverage @10 (C2, page-level) ===")
for name, ranks in rank_sets.items():
    cov = evidence_coverage(ranks, chunks_c2, k=10)
    print(f"  {name:10s} avg_pair_fraction_hit@10={round(cov['avg_pair_fraction_hit'],4)}  alongside any_hit_recall@10={round(cov['any_hit_recall'],4)} (hits {cov['any_hit_hits']}/{cov['n']})")


=== 3. Evidence-level coverage @10 (C2, page-level) ===
  BM25       avg_pair_fraction_hit@10=0.0222  alongside any_hit_recall@10=0.0222 (hits 1/45)
  Dense      avg_pair_fraction_hit@10=0.1556  alongside any_hit_recall@10=0.1556 (hits 7/45)
  Hybrid_RRF avg_pair_fraction_hit@10=0.1556  alongside any_hit_recall@10=0.1556 (hits 7/45)


In [22]:
# --- 4) Doc-routed retrieval: top-3 docs by dense doc-level score (max chunk score per doc), rank within them ---
# candidates = chunks in top-3 docs by max dense chunk score per doc
# dense-routed: candidates ranked by dense chunk score
# hybrid-routed: candidates ranked by RRF score 1/(60+rank_bm25_global)+1/(60+rank_dense_global)

def global_ranks_from_scores(scores):
    order = np.argsort(-scores)
    ranks = np.empty(len(scores), dtype=np.int32)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks

doc_ids_arr = np.array([c["meta"].get("document_id") for c in chunks_c2])

def routed_rankings(top_docs=3, topn=max(KS)):
    dense_routed = []
    hybrid_routed = []
    for qi in range(len(EVAL_QUERIES)):
        ds = dense_scores_all[qi]
        bs = bm25_scores_all[qi]
        # doc-level dense score = max chunk score per doc
        doc_best = {}
        for idx, did in enumerate(doc_ids_arr):
            s = float(ds[idx])
            if (did not in doc_best) or (s > doc_best[did]):
                doc_best[did] = s
        keep_docs = set(sorted(doc_best, key=doc_best.get, reverse=True)[:top_docs])
        cand = np.array([i for i, did in enumerate(doc_ids_arr) if did in keep_docs], dtype=np.int32)
        # dense within
        cand_d = cand[np.argsort(-ds[cand])][:topn]
        dense_routed.append([(int(i), float(ds[i])) for i in cand_d])
        # hybrid within (global RRF ranks)
        br = global_ranks_from_scores(bs)
        dr = global_ranks_from_scores(ds)
        rrf_scores = (1.0 / (RRF_K + br[cand])) + (1.0 / (RRF_K + dr[cand]))
        cand_h = cand[np.argsort(-rrf_scores)][:topn]
        hybrid_routed.append([(int(i), float(rrf_scores[np.where(cand == i)[0][0]])) for i in cand_h])
    return dense_routed, hybrid_routed

dense_routed_r, hybrid_routed_r = routed_rankings(top_docs=3, topn=max(KS))

print("=== 4. Doc-routed retrieval (top-3 docs by max dense chunk score per doc; rank within them) ===")
print("method: candidates=chunks in top-3 dense-scored docs; dense=candidates by dense score; hybrid=candidates by RRF 1/(60+global_rank_bm25)+1/(60+global_rank_dense)")
for name, rr in [("Dense_routed", dense_routed_r), ("Hybrid_routed", hybrid_routed_r)]:
    m = score_page_level(rr, chunks_c2, ks=KS)
    print(f"  {name:14s}", _fmt(m))


=== 4. Doc-routed retrieval (top-3 docs by max dense chunk score per doc; rank within them) ===
method: candidates=chunks in top-3 dense-scored docs; dense=candidates by dense score; hybrid=candidates by RRF 1/(60+global_rank_bm25)+1/(60+global_rank_dense)
  Dense_routed   {'R@1': 0.0667, 'R@3': 0.1333, 'R@5': 0.1778, 'R@10': 0.1778, 'MRR': 0.1052, 'hits': {1: 3, 3: 6, 5: 8, 10: 8}}
  Hybrid_routed  {'R@1': 0.0444, 'R@3': 0.0667, 'R@5': 0.1111, 'R@10': 0.2, 'MRR': 0.0763, 'hits': {1: 2, 3: 3, 5: 5, 10: 9}}


In [23]:
# --- 5) Split by single-evidence vs multi-evidence rows (counts) ---
single_idx = [i for i, g in enumerate(EVAL_GOLD_SETS) if len(g) == 1]
multi_idx = [i for i, g in enumerate(EVAL_GOLD_SETS) if len(g) > 1]
print("=== 5. Split by single-evidence vs multi-evidence rows (C2, page-level) ===")
print(f"counts: total={len(EVAL_ROWS)} single_evidence_rows={len(single_idx)} multi_evidence_rows={len(multi_idx)}")

def score_subset(rankings, chunks, idxs, ks=KS):
    n = len(idxs); recalls = {k: 0 for k in ks}; mrr_sum = 0.0
    for qi in idxs:
        gold = EVAL_GOLD_SETS[qi]; ranked = rankings[qi]; rr = 0.0
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if chunk_matches_gold(chunks[idx]["meta"], gold):
                rr = 1.0 / rank; break
        mrr_sum += rr
        for k in ks:
            if any(chunk_matches_gold(chunks[idx]["meta"], gold) for idx, _s in ranked[:k]):
                recalls[k] += 1
    return {"n": n, **{f"recall@{k}": (recalls[k]/n if n else 0.0) for k in ks}, **{f"hits@{k}": recalls[k] for k in ks}, "mrr": (mrr_sum/n if n else 0.0)}

for split_name, idxs in [("single_evidence", single_idx), ("multi_evidence", multi_idx)]:
    print(f"\n-- {split_name} (n={len(idxs)}) --")
    for name, ranks in rank_sets.items():
        m = score_subset(ranks, chunks_c2, idxs, ks=KS)
        print(f"  {name:10s}", _fmt(m))

print("\n--- extended diagnosis end (printed only; no existing matching rules or result files changed) ---")


=== 5. Split by single-evidence vs multi-evidence rows (C2, page-level) ===
counts: total=45 single_evidence_rows=35 multi_evidence_rows=10

-- single_evidence (n=35) --
  BM25       {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0286, 'MRR': 0.0048, 'hits': {1: 0, 3: 0, 5: 0, 10: 1}}
  Dense      {'R@1': 0.0857, 'R@3': 0.1714, 'R@5': 0.2, 'R@10': 0.2, 'MRR': 0.1295, 'hits': {1: 3, 3: 6, 5: 7, 10: 7}}
  Hybrid_RRF {'R@1': 0.0, 'R@3': 0.1143, 'R@5': 0.1714, 'R@10': 0.2, 'MRR': 0.0698, 'hits': {1: 0, 3: 4, 5: 6, 10: 7}}

-- multi_evidence (n=10) --
  BM25       {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0, 'MRR': 0.0, 'hits': {1: 0, 3: 0, 5: 0, 10: 0}}
  Dense      {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0, 'MRR': 0.0, 'hits': {1: 0, 3: 0, 5: 0, 10: 0}}
  Hybrid_RRF {'R@1': 0.0, 'R@3': 0.0, 'R@5': 0.0, 'R@10': 0.0, 'MRR': 0.0, 'hits': {1: 0, 3: 0, 5: 0, 10: 0}}

--- extended diagnosis end (printed only; no existing matching rules or result files changed) ---


## Rerank + decomposition follow-up (C2, cached, cleaned BM25)
Appended tasks 1–4. C2 only, reused cache (no re-embedding). BM25 uses cleaned tokenizer (lowercase, punctuation stripped, stopwords). No existing matching rules or result files changed.


In [24]:
# --- Follow-up setup (C2, cached, cleaned BM25 for all BM25 here) ---
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

assert 'build_chunks_for_slice' in globals()
assert 'SLICE_DOC_IDS' in globals() and 'EVAL_ROWS' in globals() and 'EVAL_GOLD_SETS' in globals()
assert 'CACHE_ROOT' in globals() and 'KS' in globals() and 'RRF_K' in globals()
assert 'dense_search' in globals() and 'rrf_fuse' in globals() and 'chunk_matches_gold' in globals()

EVAL_QUERIES = [(r.get("question_text") or r.get("question") or "") for r in EVAL_ROWS]

def _ensure_base():
    global chunks_c2, embs_c2, qembs, cache_ids, chunk_ids_now, doc_ids_arr
    if 'chunks_c2' not in globals():
        chunks_c2 = build_chunks_for_slice("C2", SLICE_DOC_IDS)
    cache_dir = Path(CACHE_ROOT) / "C2"
    embs_c2 = np.load(cache_dir / "embeddings.npy")
    cache_ids = json.loads((cache_dir / "chunk_ids.json").read_text(encoding="utf-8"))
    qembs = np.load(Path(CACHE_ROOT) / "eval_query_embeddings.npy")
    chunk_ids_now = [c["chunk_id"] for c in chunks_c2]
    assert cache_ids == chunk_ids_now
    assert qembs.shape[0] == len(EVAL_ROWS) == len(EVAL_QUERIES)
    doc_ids_arr = np.array([c["meta"].get("document_id") for c in chunks_c2])

_ensure_base()

from rank_bm25 import BM25Okapi
STOPWORDS_SIMPLE = set("""
a an and are as at be by for from has have he in is it its of on or that the to was were will with this these those but if into no not than then so such can may we our their they you your i me my
""".split())

def simple_tok(text):
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    toks = text.split()
    return [t for t in toks if t not in STOPWORDS_SIMPLE and len(t) > 1]

if 'bm25_simple' not in globals() or 'corpus_simple' not in globals():
    corpus_simple = [simple_tok(c.get("text") or "") for c in chunks_c2]
    bm25_simple = BM25Okapi(corpus_simple)

def topk_from_scores(scores, k):
    k = min(k, len(scores))
    if k <= 0: return []
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]

bm25s_scores_all = [bm25_simple.get_scores(simple_tok(q or "")) for q in EVAL_QUERIES]
dense_scores_all = [embs_c2 @ qembs[i] for i in range(len(EVAL_QUERIES))]

def score_page_level(rankings, chunks, ks):
    n = len(EVAL_ROWS); recalls = {k: 0 for k in ks}; mrr_sum = 0.0
    for qi, gold in enumerate(EVAL_GOLD_SETS):
        ranked = rankings[qi]; rr = 0.0
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if chunk_matches_gold(chunks[idx]["meta"], gold):
                rr = 1.0 / rank; break
        mrr_sum += rr
        for k in ks:
            if any(chunk_matches_gold(chunks[idx]["meta"], gold) for idx, _s in ranked[:k]):
                recalls[k] += 1
    return {"n": n, **{f"recall@{k}": recalls[k]/n for k in ks}, **{f"hits@{k}": recalls[k] for k in ks}, "mrr": mrr_sum/n}

def evidence_coverage(rankings, chunks, k=10):
    per_row = []
    for qi, gold in enumerate(EVAL_GOLD_SETS):
        ranked = rankings[qi][:k]
        if not gold:
            per_row.append(0.0); continue
        hit_pairs = 0
        for g in gold:
            if any(chunk_matches_gold(chunks[idx]["meta"], {g}) for idx, _s in ranked):
                hit_pairs += 1
        per_row.append(hit_pairs / len(gold))
    return {"n": len(per_row), "avg_pair_fraction_hit": sum(per_row)/len(per_row) if per_row else 0.0, "per_row": per_row}

single_idx = [i for i, g in enumerate(EVAL_GOLD_SETS) if len(g) == 1]
multi_idx = [i for i, g in enumerate(EVAL_GOLD_SETS) if len(g) > 1]

def score_subset(rankings, chunks, idxs, ks):
    n = len(idxs); recalls = {k: 0 for k in ks}; mrr_sum = 0.0
    for qi in idxs:
        gold = EVAL_GOLD_SETS[qi]; ranked = rankings[qi]; rr = 0.0
        for rank, (idx, _s) in enumerate(ranked, start=1):
            if chunk_matches_gold(chunks[idx]["meta"], gold):
                rr = 1.0 / rank; break
        mrr_sum += rr
        for k in ks:
            if any(chunk_matches_gold(chunks[idx]["meta"], gold) for idx, _s in ranked[:k]):
                recalls[k] += 1
    return {"n": n, **{f"recall@{k}": (recalls[k]/n if n else 0.0) for k in ks}, **{f"hits@{k}": recalls[k] for k in ks}, "mrr": (mrr_sum/n if n else 0.0)}

def _fmt(m, ks):
    d = {f"R@{k}": round(m[f'recall@{k}'], 4) for k in ks}
    d["MRR"] = round(m['mrr'], 4); d["hits"] = {k: m[f'hits@{k}'] for k in ks}; d["n"] = m["n"]
    return d

print("follow-up setup ready: C2", len(chunks_c2), "eval", len(EVAL_ROWS), "single", len(single_idx), "multi", len(multi_idx))
print("BM25 (all here): cleaned tokenizer lower+punct-strip+stopwords n=", len(STOPWORDS_SIMPLE))


follow-up setup ready: C2 50221 eval 45 single 35 multi 10
BM25 (all here): cleaned tokenizer lower+punct-strip+stopwords n= 50


In [25]:
# --- 1) Hybrid RRF (cleaned BM25 + dense), candidate pool top-100 ---
POOL_K = 100
REPORT_KS = (1,3,5,10,20,50,100)

bm25_clean_rank100 = [topk_from_scores(bm25s_scores_all[i], POOL_K) for i in range(len(EVAL_QUERIES))]
dense_rank100 = [topk_from_scores(dense_scores_all[i], POOL_K) for i in range(len(EVAL_QUERIES))]
hybrid_clean_rank100 = [rrf_fuse([bm25_clean_rank100[i], dense_rank100[i]], k=RRF_K, topn=POOL_K) for i in range(len(EVAL_QUERIES))]

print("=== 1. Hybrid RRF (cleaned BM25 + dense), pool top-100 (C2, page-level) ===")
m1 = score_page_level(hybrid_clean_rank100, chunks_c2, ks=REPORT_KS)
print("Hybrid_RRF_clean", _fmt(m1, REPORT_KS))


=== 1. Hybrid RRF (cleaned BM25 + dense), pool top-100 (C2, page-level) ===
Hybrid_RRF_clean {'R@1': 0.0, 'R@3': 0.0444, 'R@5': 0.0667, 'R@10': 0.1111, 'R@20': 0.2444, 'R@50': 0.3778, 'R@100': 0.6444, 'MRR': 0.0474, 'hits': {1: 0, 3: 2, 5: 3, 10: 5, 20: 11, 50: 17, 100: 29}, 'n': 45}


In [25]:
# --- 2) Cross-encoder rerank of top-100 pool (cache scores to disk) ---
RERANK_CACHE = Path(CACHE_ROOT) / "C2" / "rerank_scores.npy"
RERANK_META = Path(CACHE_ROOT) / "C2" / "rerank_scores.meta.json"
RERANK_LAT = Path(CACHE_ROOT) / "C2" / "rerank_latency.json"

rerank_model_name = None
rerank_scores = None
latencies = []

want_meta = {"pool_k": POOL_K, "n_queries": len(EVAL_QUERIES), "bm25": "cleaned", "dense": "bge-small-en-v1.5", "rrf_k": RRF_K}

def _load_or_run_rerank():
    global rerank_model_name, rerank_scores, latencies
    if RERANK_CACHE.exists() and RERANK_META.exists() and RERANK_LAT.exists():
        meta = json.loads(RERANK_META.read_text(encoding="utf-8"))
        if meta.get("want") == want_meta and meta.get("shape") == [len(EVAL_QUERIES), POOL_K]:
            arr = np.load(RERANK_CACHE)
            if arr.shape == (len(EVAL_QUERIES), POOL_K):
                rerank_scores = arr
                rerank_model_name = meta.get("model")
                lat = json.loads(RERANK_LAT.read_text(encoding="utf-8"))
                latencies = lat.get("per_query_seconds", [])
                print("rerank cache hit:", rerank_model_name, rerank_scores.shape)
                return
    # try BAAI/bge-reranker-base, fallback to cached ms-marco
    try:
        from fastembed.rerank.cross_encoder import TextCrossEncoder
        model = None; used = None; last_err = None
        for name in ["BAAI/bge-reranker-base", "Xenova/ms-marco-MiniLM-L-6-v2"]:
            try:
                model = TextCrossEncoder(model_name=name)
                used = name
                break
            except Exception as e:
                last_err = e
        if model is None:
            raise RuntimeError(f"no reranker available: {last_err}")
        rerank_model_name = used
        scores_out = np.full((len(EVAL_QUERIES), POOL_K), -1e9, dtype=np.float32)
        latencies = []
        for qi in range(len(EVAL_QUERIES)):
            pool = hybrid_clean_rank100[qi]
            texts = [(chunks_c2[idx].get("text") or "") for idx, _ in pool]
            t0 = time.perf_counter()
            sc = list(model.rerank(EVAL_QUERIES[qi], texts))
            t1 = time.perf_counter()
            latencies.append(t1 - t0)
            for j, s in enumerate(sc):
                scores_out[qi, j] = float(s)
            if (qi+1) % 10 == 0 or qi == 0 or qi == len(EVAL_QUERIES)-1:
                print(f"rerank progress {qi+1}/{len(EVAL_QUERIES)}", flush=True)
        rerank_scores = scores_out
        RERANK_CACHE.parent.mkdir(parents=True, exist_ok=True)
        np.save(RERANK_CACHE, rerank_scores)
        RERANK_META.write_text(json.dumps({"model": rerank_model_name, "want": want_meta, "shape": list(rerank_scores.shape)}, sort_keys=True), encoding="utf-8")
        RERANK_LAT.write_text(json.dumps({"per_query_seconds": latencies, "mean_seconds": (sum(latencies)/len(latencies) if latencies else None)}, sort_keys=True), encoding="utf-8")
        print("rerank built+cached:", rerank_model_name, rerank_scores.shape)
    except Exception as e:
        rerank_model_name = None
        rerank_scores = None
        latencies = []
        print("rerank unavailable:", repr(e))

_load_or_run_rerank()

reranked_rank100 = None
if rerank_scores is not None:
    reranked_rank100 = []
    for qi in range(len(EVAL_QUERIES)):
        pool = hybrid_clean_rank100[qi]
        order = np.argsort(-rerank_scores[qi])
        reranked_rank100.append([(pool[j][0], float(rerank_scores[qi, j])) for j in order])
    print("\n=== 2. Cross-encoder rerank of top-100 pool (C2, page-level) ===")
    print("reranker model:", rerank_model_name)
    m2 = score_page_level(reranked_rank100, chunks_c2, ks=(1,3,5,10))
    print("Reranked", _fmt(m2, (1,3,5,10)))
    if latencies:
        print("latency per query sec: mean", round(sum(latencies)/len(latencies), 4), "min", round(min(latencies),4), "max", round(max(latencies),4), "n", len(latencies))
else:
    print("\n=== 2. Cross-encoder rerank: SKIPPED (unavailable) ===")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

rerank progress 1/45
rerank progress 10/45
rerank progress 20/45
rerank progress 30/45
rerank progress 40/45
rerank progress 45/45
rerank built+cached: BAAI/bge-reranker-base (45, 100)

=== 2. Cross-encoder rerank of top-100 pool (C2, page-level) ===
reranker model: BAAI/bge-reranker-base
Reranked {'R@1': 0.0444, 'R@3': 0.1111, 'R@5': 0.2, 'R@10': 0.2222, 'MRR': 0.1088, 'hits': {1: 2, 3: 5, 5: 9, 10: 10}, 'n': 45}
latency per query sec: mean 90.8145 min 68.5087 max 145.695 n 45


In [ ]:
# --- 3) Decomposition for multi-evidence rows (10): LLM sub-queries (cached, no gold doc ids in prompt) ---
# Edited: per-subq rerank top-20 only (cached by subq+pool), row-wise cache save/skip, OMP_NUM_THREADS for onnxruntime, then T4.
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

# 4) set onnxruntime thread count from OMP_NUM_THREADS (best-effort)
_omp = os.environ.get("OMP_NUM_THREADS")
if _omp:
    try:
        _n = max(1, int(_omp))
        os.environ.setdefault("ORT_INTRA_OP_NUM_THREADS", str(_n))
        try:
            import onnxruntime as ort
            ort.set_default_logger_severity(3)
            # global defaults where supported
            try:
                ort.intra_op_num_threads = _n
            except Exception:
                pass
        except Exception:
            pass
    except Exception:
        _n = None
else:
    _n = None

DECOMP_CACHE = Path(CACHE_ROOT) / "C2" / "decomp_cache.json"
SUBQ_EMB_CACHE = Path(CACHE_ROOT) / "C2" / "subquery_embeddings.json"
SUBQ_RERANK_CACHE = Path(CACHE_ROOT) / "C2" / "subquery_rerank_cache.json"
RERANK_TOPK_SUBQ = 20  # per sub-query rerank only top-20 of hybrid pool
POOL_K = globals().get("POOL_K", 100)
REPORT_KS = globals().get("REPORT_KS", (1,3,5,10,20,50,100))

def _load_json_cache(p, default):
    p = Path(p)
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return default
    return default

def _save_json_cache(p, d):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(d, indent=2, sort_keys=True), encoding="utf-8")

decomp_cache = _load_json_cache(DECOMP_CACHE, {})
subq_emb_cache = _load_json_cache(SUBQ_EMB_CACHE, {})
subq_rerank_cache = _load_json_cache(SUBQ_RERANK_CACHE, {})

def _row_key(question, n_parts):
    return hashlib.sha256((question + "||" + str(n_parts)).encode("utf-8")).hexdigest()

def llm_decompose(question, n_parts, cache, key):
    if key in cache and isinstance(cache[key], dict) and cache[key].get("sub_queries"):
        return cache[key]["sub_queries"], "cache"
    try:
        from src.llm import client as llm_client
    except Exception as e:
        return None, f"import_error:{e!r}"
    prompt = (
        "Split the following financial question into exactly " + str(n_parts) + " focused sub-questions, "
        "one per company/document being compared or referenced. Do not mention document IDs. "
        "Return ONLY a JSON array of strings.\nQuestion: " + question
    )
    messages = [{"role": "user", "content": prompt}]
    try:
        resp = llm_client.chat(messages, max_tokens=300, temperature=0.0, cache=True, timeout=60.0)
        txt = resp.text
        m = re.search(r"\[.*\]", txt, re.DOTALL)
        arr = json.loads(m.group(0) if m else txt)
        subs = [str(x).strip() for x in arr if str(x).strip()]
        subs = subs[:n_parts]
        if len(subs) == n_parts:
            cache[key] = {**(cache.get(key) or {}), "question": question, "n_parts": n_parts, "sub_queries": subs}
            _save_json_cache(DECOMP_CACHE, cache)
            return subs, "llm"
        return None, f"bad_count:{len(subs)}"
    except Exception as e:
        return None, f"error:{e!r}"

def embed_subq(subq, cache):
    skey = hashlib.sha256(("dense:"+subq).encode("utf-8")).hexdigest()
    if skey in cache:
        return np.asarray(cache[skey], dtype=np.float32), "cache"
    try:
        base_url = os.environ.get("LLM_EMBEDDINGS_URL") or os.environ.get("EMBEDDINGS_URL")
        api_key = os.environ.get("EMBEDDINGS_API_KEY") or os.environ.get("LLM_EMBEDDINGS_API_KEY") or ""
        ca_cert = os.environ.get("EMBEDDINGS_CA_CERT") or ""
        if 'embed_texts_vm02' in globals():
            arr = embed_texts_vm02([subq or ""], base_url, api_key, EMBED_MODEL if 'EMBED_MODEL' in globals() else "bge-small-en-v1.5", ca_cert, batch=1, max_retries=5, label="subq")[0]
        else:
            raise RuntimeError("embed_texts_vm02 not available")
        cache[skey] = [float(x) for x in arr]
        _save_json_cache(SUBQ_EMB_CACHE, cache)
        return np.asarray(arr, dtype=np.float32), "embed"
    except Exception as e:
        return None, f"error:{e!r}"

# reusable reranker model for subquery rerank (not re-ranking existing T2 cached work)
_subq_rerank_model = globals().get('_subq_rerank_model')

def _get_subq_rerank_model():
    global _subq_rerank_model
    if _subq_rerank_model is not None:
        return _subq_rerank_model
    model_name = globals().get('rerank_model_name')
    if model_name is None:
        # try load from T2 cache meta
        meta_p = Path(CACHE_ROOT) / "C2" / "rerank_scores.meta.json"
        if meta_p.exists():
            try:
                model_name = json.loads(meta_p.read_text(encoding="utf-8")).get("model")
            except Exception:
                model_name = None
    if model_name is None:
        return None
    try:
        from fastembed.rerank.cross_encoder import TextCrossEncoder
        kwargs = {"model_name": model_name}
        if _n is not None:
            try:
                _subq_rerank_model = TextCrossEncoder(model_name=model_name, threads=_n)
            except TypeError:
                _subq_rerank_model = TextCrossEncoder(model_name=model_name)
        else:
            _subq_rerank_model = TextCrossEncoder(model_name=model_name)
        globals()['_subq_rerank_model'] = _subq_rerank_model
        return _subq_rerank_model
    except Exception as e:
        print("subq rerank model load failed:", e, flush=True)
        return None

def _subq_rerank_scores(subq, pool_rerank):
    # pool_rerank: list[(idx, score)] length<=RERANK_TOPK_SUBQ; cache by (subq, pool_size, pool ids)
    pool_idx = [int(i) for i, _s in pool_rerank]
    rkey = hashlib.sha256((subq + "||ps=" + str(RERANK_TOPK_SUBQ) + "||pool=" + ",".join(map(str, pool_idx))).encode("utf-8")).hexdigest()
    if rkey in subq_rerank_cache:
        ent = subq_rerank_cache[rkey]
        if ent.get("pool_idx") == pool_idx and len(ent.get("scores", [])) == len(pool_idx):
            return [float(x) for x in ent["scores"]], "cache"
    model = _get_subq_rerank_model()
    if model is None:
        return None, "no_model"
    try:
        texts = [(chunks_c2[idx].get("text") or "") for idx in pool_idx]
        sc = [float(x) for x in list(model.rerank(subq, texts))]
        subq_rerank_cache[rkey] = {"subq": subq, "pool_size": RERANK_TOPK_SUBQ, "pool_idx": pool_idx, "scores": sc}
        _save_json_cache(SUBQ_RERANK_CACHE, subq_rerank_cache)
        return sc, "rerank"
    except Exception as e:
        return None, f"error:{e!r}"

def decomp_retrieve_merge(subqs):
    merged = {}
    per_sub = []
    for sq in subqs:
        bs = bm25_simple.get_scores(simple_tok(sq or ""))
        b_rank = topk_from_scores(bs, POOL_K)
        qv, qv_src = embed_subq(sq, subq_emb_cache)
        if qv is not None:
            ds = embs_c2 @ qv
            d_rank = topk_from_scores(ds, POOL_K)
            pool_full = rrf_fuse([b_rank, d_rank], k=RRF_K, topn=POOL_K)
        else:
            pool_full = b_rank
        pool_rerank = pool_full[:RERANK_TOPK_SUBQ]
        sc, sc_src = _subq_rerank_scores(sq, pool_rerank)
        if sc is not None:
            for j, (idx, _s) in enumerate(pool_rerank):
                score = float(sc[j])
                if (idx not in merged) or (score > merged[idx]):
                    merged[idx] = score
        else:
            # fallback: RRF scores for top-20 only
            for idx, s in pool_rerank:
                score = float(s)
                if (idx not in merged) or (score > merged[idx]):
                    merged[idx] = score
        per_sub.append({"subq": sq, "dense": qv_src, "rerank": sc_src, "pool_n": len(pool_full), "rerank_n": len(pool_rerank)})
    merged_ranked = sorted(merged.items(), key=lambda x: -x[1])
    return [(int(i), float(s)) for i, s in merged_ranked], per_sub

print("=== 3. Decomposition for multi-evidence rows (10) ===", flush=True)
print(f"config: per-subq rerank top-{RERANK_TOPK_SUBQ} of hybrid pool (cached), save after each row, skip if merged cached (ps={RERANK_TOPK_SUBQ}); OMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS')}", flush=True)
multi_results = {}
decomp_errors = []

if not multi_idx:
    print("no multi-evidence rows", flush=True)
else:
    for qi in multi_idx:
        row = EVAL_ROWS[qi]
        q = row.get("question_text") or row.get("question") or ""
        n_parts = len(EVAL_GOLD_SETS[qi])
        key = _row_key(q, n_parts)
        ent = decomp_cache.get(key) or {}
        # skip row if fully done in cache with current params
        if ent.get("merged_ranked") and ent.get("decomp_rerank_pool_size") == RERANK_TOPK_SUBQ:
            merged_ranked = [(int(i), float(s)) for i, s in ent["merged_ranked"]]
            multi_results[qi] = {"sub_queries": ent.get("sub_queries", []), "merged_ranked": merged_ranked, "status": "cache_row"}
            print(f"row {qi}: skipped (cached merged, ps={RERANK_TOPK_SUBQ})", flush=True)
            continue
        subs = ent.get("sub_queries") if ent.get("sub_queries") else None
        if subs is not None:
            status = "cache_subq"
        else:
            print(f"row {qi}: decomposing (n_parts={n_parts})", flush=True)
            subs, status = llm_decompose(q, n_parts, decomp_cache, key)
        if subs is None:
            decomp_errors.append((qi, status))
            print(f"  row {qi} decomp fail: {status}", flush=True)
            continue
        print(f"row {qi}: merge+rerank top-{RERANK_TOPK_SUBQ} per subq (n_subq={len(subs)}, src={status})", flush=True)
        merged_ranked, per_sub = decomp_retrieve_merge(subs)
        # save after each row
        ent = decomp_cache.get(key) or {}
        ent.update({
            "question": q, "n_parts": n_parts, "sub_queries": subs,
            "merged_ranked": [[int(i), float(s)] for i, s in merged_ranked],
            "decomp_rerank_pool_size": RERANK_TOPK_SUBQ,
            "per_sub": per_sub,
        })
        decomp_cache[key] = ent
        _save_json_cache(DECOMP_CACHE, decomp_cache)
        multi_results[qi] = {"sub_queries": subs, "merged_ranked": merged_ranked, "status": status}
        print(f"  row {qi}: saved (merged_n={len(merged_ranked)})", flush=True)

    if decomp_errors and not multi_results:
        print("decomposition unavailable/failed for all multi-evidence rows; coverage@10 not computed.", flush=True)
        for qi, st in decomp_errors:
            print(f"  row idx {qi}: {st}", flush=True)
    else:
        if decomp_errors:
            print("decomposition errors (skipped):", decomp_errors, flush=True)
        done_idxs = [qi for qi in multi_idx if qi in multi_results]
        per_row_cov = []
        for qi in done_idxs:
            gold = EVAL_GOLD_SETS[qi]
            ranked = multi_results[qi]["merged_ranked"][:10]
            if not gold:
                per_row_cov.append(0.0); continue
            hit_pairs = sum(1 for g in gold if any(chunk_matches_gold(chunks_c2[idx]["meta"], {g}) for idx, _s in ranked))
            per_row_cov.append(hit_pairs / len(gold))
        avg_cov = sum(per_row_cov)/len(per_row_cov) if per_row_cov else 0.0
        print(f"multi-evidence rows total={len(multi_idx)} completed={len(done_idxs)}", flush=True)
        print(f"evidence-level coverage@10 (avg pair fraction hit, completed multi-evidence rows): {round(avg_cov,4)}", flush=True)
        for qi in done_idxs:
            print(f"  row {qi} n_gold={len(EVAL_GOLD_SETS[qi])} cov@10={round(per_row_cov[done_idxs.index(qi)],4)} subq_src={multi_results[qi]['status']}", flush=True)

# ensure T1/T2 inputs for T4 (no re-embed/re-rank)
if 'hybrid_clean_rank100' not in globals():
    try:
        bm25_clean_rank100 = [topk_from_scores(bm25s_scores_all[i], POOL_K) for i in range(len(EVAL_QUERIES))]
        dense_rank100 = [topk_from_scores(dense_scores_all[i], POOL_K) for i in range(len(EVAL_QUERIES))]
        hybrid_clean_rank100 = [rrf_fuse([bm25_clean_rank100[i], dense_rank100[i]], k=RRF_K, topn=POOL_K) for i in range(len(EVAL_QUERIES))]
        print("built hybrid_clean_rank100 for T4 (from cached scores)", flush=True)
    except Exception as _e:
        print("T4 setup failed (run cell 26/27):", _e, flush=True)
        hybrid_clean_rank100 = None
if 'reranked_rank100' not in globals() or reranked_rank100 is None:
    try:
        _rc = Path(CACHE_ROOT) / "C2" / "rerank_scores.npy"
        _rm = Path(CACHE_ROOT) / "C2" / "rerank_scores.meta.json"
        _rl = Path(CACHE_ROOT) / "C2" / "rerank_latency.json"
        if _rc.exists() and _rm.exists() and hybrid_clean_rank100 is not None:
            _arr = np.load(_rc)
            if _arr.shape[0] == len(EVAL_QUERIES):
                rerank_model_name = globals().get('rerank_model_name') or json.loads(_rm.read_text(encoding="utf-8")).get("model")
                if 'latencies' not in globals() or not latencies:
                    if _rl.exists():
                        latencies = json.loads(_rl.read_text(encoding="utf-8")).get("per_query_seconds", [])
                reranked_rank100 = []
                for _qi in range(len(EVAL_QUERIES)):
                    _pool = hybrid_clean_rank100[_qi]
                    _order = np.argsort(-_arr[_qi])
                    reranked_rank100.append([(_pool[_j][0], float(_arr[_qi, _j])) for _j in _order])
                print("loaded T2 rerank from cache for T4", flush=True)
        elif hybrid_clean_rank100 is None:
            reranked_rank100 = None
    except Exception as _e:
        print("T2 cache load for T4 failed:", _e, flush=True)
        reranked_rank100 = None

# --- 4) Split every table by single vs multi-evidence, with counts (T4) ---
print("\n=== 4. Split every table by single vs multi-evidence, with counts ===", flush=True)
print(f"counts: total={len(EVAL_ROWS)} single_evidence_rows={len(single_idx)} multi_evidence_rows={len(multi_idx)}", flush=True)

print("\n[Table 1] Hybrid RRF (cleaned BM25 + dense), pool top-100 (page-level)", flush=True)
for split_name, idxs in [("single_evidence", single_idx), ("multi_evidence", multi_idx), ("all", list(range(len(EVAL_ROWS))))]:
    ms = score_subset(hybrid_clean_rank100, chunks_c2, idxs, ks=REPORT_KS)
    print(f"  {split_name} n={ms['n']}", _fmt(ms, REPORT_KS), flush=True)

print("\n[Table 2] Cross-encoder rerank of top-100 pool (page-level)", flush=True)
if reranked_rank100 is not None:
    print("  model:", rerank_model_name if 'rerank_model_name' in globals() else None, flush=True)
    for split_name, idxs in [("single_evidence", single_idx), ("multi_evidence", multi_idx), ("all", list(range(len(EVAL_ROWS))))]:
        ms = score_subset(reranked_rank100, chunks_c2, idxs, ks=(1,3,5,10))
        print(f"  {split_name} n={ms['n']}", _fmt(ms, (1,3,5,10)), flush=True)
    if 'latencies' in globals() and latencies:
        print("  latency/query sec mean/min/max:", round(sum(latencies)/len(latencies),4), round(min(latencies),4), round(max(latencies),4), flush=True)
else:
    print("  SKIPPED/unavailable", flush=True)

print("\n[Table 3] Decomposition multi-evidence evidence-level coverage@10", flush=True)
if not multi_idx:
    print("  no multi-evidence rows", flush=True)
elif multi_results:
    done_idxs = [qi for qi in multi_idx if qi in multi_results]
    per_row_cov = []
    for qi in done_idxs:
        gold = EVAL_GOLD_SETS[qi]; ranked = multi_results[qi]["merged_ranked"][:10]
        hit_pairs = sum(1 for g in gold if any(chunk_matches_gold(chunks_c2[idx]["meta"], {g}) for idx,_s in ranked)) if gold else 0
        per_row_cov.append((hit_pairs/len(gold)) if gold else 0.0)
    print(f"  multi_evidence n={len(multi_idx)} completed={len(done_idxs)} avg_pair_fraction_hit@10={round(sum(per_row_cov)/len(per_row_cov),4) if per_row_cov else 'NA'}", flush=True)
    print(f"  single_evidence n={len(single_idx)} (decomposition N/A by design)", flush=True)
else:
    print(f"  multi_evidence n={len(multi_idx)} completed=0 coverage@10=NA ({'errors: '+str(decomp_errors) if decomp_errors else 'unavailable'})", flush=True)
    print(f"  single_evidence n={len(single_idx)} (decomposition N/A)", flush=True)

print("\n--- follow-up end (printed only; no existing matching rules or result files changed) ---", flush=True)


=== 3. Decomposition for multi-evidence rows (10) ===
config: per-subq rerank top-20 of hybrid pool (cached), save after each row, skip if merged cached (ps=20); OMP_NUM_THREADS=None
row 0: merge+rerank top-20 per subq (n_subq=2, src=cache_subq)


In [ ]:
# --- 4) Split every table by single vs multi-evidence, with counts ---
print("=== 4. Split every table by single vs multi-evidence, with counts ===")
print(f"counts: total={len(EVAL_ROWS)} single_evidence_rows={len(single_idx)} multi_evidence_rows={len(multi_idx)}")

# Table 1 (task 1)
print("\n[Table 1] Hybrid RRF (cleaned BM25 + dense), pool top-100 (page-level)")
m1t = score_page_level(hybrid_clean_rank100, chunks_c2, ks=REPORT_KS)
for split_name, idxs in [("single_evidence", single_idx), ("multi_evidence", multi_idx), ("all", list(range(len(EVAL_ROWS))))]:
    ms = score_subset(hybrid_clean_rank100, chunks_c2, idxs, ks=REPORT_KS)
    print(f"  {split_name} n={ms['n']}", _fmt(ms, REPORT_KS))

# Table 2 (task 2)
print("\n[Table 2] Cross-encoder rerank of top-100 pool (page-level)")
if reranked_rank100 is not None:
    print("  model:", rerank_model_name)
    for split_name, idxs in [("single_evidence", single_idx), ("multi_evidence", multi_idx), ("all", list(range(len(EVAL_ROWS))))]:
        ms = score_subset(reranked_rank100, chunks_c2, idxs, ks=(1,3,5,10))
        print(f"  {split_name} n={ms['n']}", _fmt(ms, (1,3,5,10)))
    if latencies:
        print("  latency/query sec mean/min/max:", round(sum(latencies)/len(latencies),4), round(min(latencies),4), round(max(latencies),4))
else:
    print("  SKIPPED/unavailable")

# Table 3 (task 3)
print("\n[Table 3] Decomposition multi-evidence evidence-level coverage@10")
if not multi_idx:
    print("  no multi-evidence rows")
elif 'multi_results' in globals() and multi_results:
    done_idxs = [qi for qi in multi_idx if qi in multi_results]
    per_row_cov = []
    for qi in done_idxs:
        gold = EVAL_GOLD_SETS[qi]; ranked = multi_results[qi]["merged_ranked"][:10]
        hit_pairs = sum(1 for g in gold if any(chunk_matches_gold(chunks_c2[idx]["meta"], {g}) for idx,_s in ranked)) if gold else 0
        per_row_cov.append((hit_pairs/len(gold)) if gold else 0.0)
    print(f"  multi_evidence n={len(multi_idx)} completed={len(done_idxs)} avg_pair_fraction_hit@10={round(sum(per_row_cov)/len(per_row_cov),4) if per_row_cov else 'NA'}")
    print(f"  single_evidence n={len(single_idx)} (decomposition N/A by design)")
else:
    print(f"  multi_evidence n={len(multi_idx)} completed=0 coverage@10=NA (unavailable)")
    print(f"  single_evidence n={len(single_idx)} (decomposition N/A)")

print("\n--- follow-up end (printed only; no existing matching rules or result files changed) ---")
